In [7]:
%matplotlib tk
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
import numpy as np
import matplotlib.pyplot as plt
import torch

from src.CocoDataset import CocoDataset
from src.fueradeJuego import direccionAtaque, ultimoDefensa, lineaFueraDeJuego
from src.viz import dibujarEscena
from src.pipeline import analizarImagen, descriptores
from src.geometria import puntoApoyo, aMetros, calcularHomografia
from src.calibracion import marcarJugadores, marcarPuntos
from src.equipos import clasificarPorSemillas
from src.modelo import crearModelo
from torchvision.transforms.functional import to_tensor
from src.campo import enCampo

ds = CocoDataset("../data/raw/football-players/valid")

modelo = crearModelo(congelarBackbone=False,ligero=False)
modelo.load_state_dict(torch.load("../outputs/modelo_experimento_ResNet_20ep.pt", map_location=torch.device('cpu')))
modelo.eval()

FasterRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(800,), max_size=1333, mode='bilinear')
  )
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): FrozenBatchNorm2d(64, eps=0.0)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): FrozenBatchNorm2d(64, eps=0.0)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): FrozenBatchNorm2d(64, eps=0.0)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): FrozenBatchNorm2d(256, eps=0.0)
          (relu): ReLU(

In [9]:
IMG_ID = 41
UMBRAL = 0.50

with torch.no_grad():
    pred = modelo([to_tensor(ds.imagen(IMG_ID))])[0]

cajas = [b for b, l, s in zip(pred["boxes"].tolist(),
                              pred["labels"].tolist(),
                              pred["scores"].tolist())
         if int(l) == 3 and s >= UMBRAL]

print(len(cajas), "jugadores")

23 jugadores


In [10]:
atacante, defensor = marcarJugadores(ds.imagen(IMG_ID), cajas)
pares     = analizarImagen(ds, IMG_ID, cajas, atacante, defensor)
etiquetas = np.array([e for _, e in pares])
print(np.unique(etiquetas, return_counts=True))

Pincha primero a un atacante y luego a un defensor cualquiera.
(array(['ata', 'def', 'dudoso'], dtype='<U6'), array([ 5, 11,  7]))


In [11]:
orden = ("corner_izq_lejano", "area_izq_lejana", "penalti_izq", "medio_lejano")
H, Hinv = calcularHomografia(marcarPuntos(ds.imagen(IMG_ID), orden))

In [12]:
metros = aMetros(H, [puntoApoyo(c) for c in cajas])

dentro    = np.array([enCampo(m) for m in metros])
metros    = metros[dentro]
etiquetas = etiquetas[dentro]

sentido = direccionAtaque("izquierda")
iDef    = ultimoDefensa(etiquetas, sentido, metros)
xLinea  = lineaFueraDeJuego(metros, iDef)

fig, ax = plt.subplots(figsize=(14, 9))
dibujarEscena(ax, metros, etiquetas, xLinea, iDef)
if xLinea is not None:
    ax.set_title(f"linea en x = {xLinea:.1f} m")

fig.savefig("../outputs/pruebaFinal.png", dpi=120, bbox_inches="tight")


In [18]:
from src.fueradeJuego import veredicto, dudososEnRiesgo

resultado = veredicto(metros, etiquetas, sentido, xLinea)
riesgo    = dudososEnRiesgo(metros, etiquetas, sentido, xLinea)

fuera = [r for r in resultado if r[2] == "fuera de juego"]

if fuera:
    print(f"{len(fuera)} jugador(es) en fuera de juego:")
    for i, m, _ in fuera:
        print(f"   jugador {i}  +{m:.2f} m")
else:
    print("ningun atacante en fuera de juego")

for i, m, e in resultado:
    if e == "ajustado":
        print(f"   jugador {i}  {m:+.2f} m  ajustado, por debajo del margen de error")

if riesgo:
    print(f"\n⚠ {len(riesgo)} jugador(es) sin equipo por detras de la linea: {riesgo}")

dudosos = np.flatnonzero(etiquetas == "dudoso")
delante = [int(i) for i in dudosos if sentido * (metros[i, 0] - xLinea) > 0]
print(f"\n⚠ {len(delante)} dudosos por delante de la linea: {delante}")

ningun atacante en fuera de juego

⚠ 6 jugador(es) sin equipo por detras de la linea: [2, 3, 6, 11, 16, 17]

⚠ 1 dudosos por delante de la linea: [13]
